# BioMQM QA with NLLB Backtranslations (Git Version)

This notebook runs the Question Answering (QA) stage using data from the repository.
**Prerequisite**: You must have pushed your local changes (including `nllb_qg_merged.jsonl`) to your GitHub repository.

### Steps:
1.  **Setup**: Clones YOUR repository and installs dependencies.
2.  **Run**: Executes QA for all languages and saves results.

In [ ]:
# ==========================================
# CONFIGURATION
# ==========================================
# Replace this with YOUR new repository URL
REPO_URL = "https://github.com/laurabon/AskQE_DNLP_2025-2026.git"
REPO_NAME = REPO_URL.split('/')[-1].replace('.git', '')
# ==========================================

## 1. Setup Environment

In [ ]:
import os
import sys
import subprocess
from google.colab import drive, files

# 1. Clone Repository
if not os.path.exists(f'/content/{REPO_NAME}'):
    print(f"Cloning {REPO_URL}...")
    result = subprocess.run(['git', 'clone', REPO_URL], capture_output=True, text=True)
    if result.returncode != 0:
        print("Error cloning repo:", result.stderr)
        print("Make sure the repo allows public access or uses a token.")
    else:
        print("Repository cloned successfully.")
else:
    print("Repository already exists.")

# 2. Install Dependencies
print("Installing dependencies (transformers, torch, accelerate, protobuf, sentencepiece)...")
!pip install -q -U transformers torch accelerate protobuf sentencepiece

# 3. Mount Drive (Optional)
MOUNT_DRIVE = True
if MOUNT_DRIVE:
    drive.mount('/content/drive')
    DRIVE_SAVE_PATH = '/content/drive/MyDrive/AskQE_NLLB_Results'
    os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)
    print(f"Results will be copied to: {DRIVE_SAVE_PATH}")

## 2. Run Execution

In [ ]:
import shutil

LANGUAGES = ['de', 'es', 'fr', 'ru', 'zh-CN']

# Paths based on repo structure
PROJECT_ROOT = f'/content/{REPO_NAME}'
QA_SCRIPT = os.path.join(PROJECT_ROOT, 'results Qwen3B baseline', 'biomqm', 'direct-prompting', 'code', 'qwen-3b-direct-prompting.py')
MERGED_INPUT = os.path.join(PROJECT_ROOT, 'results Qwen3B baseline', 'backtranslation', 'nllb_qg_merged.jsonl')
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'results Qwen3B baseline', 'biomqm', 'nllb', 'QA')

if not os.path.exists(MERGED_INPUT):
    print(f"ERROR: Input file not found at {MERGED_INPUT}")
    print("Did you push 'nllb_qg_merged.jsonl' to your new repository?")
    # List files to help debug
    print("\n--- DEBUG: File Listing ---")
    if os.path.exists(os.path.dirname(MERGED_INPUT)):
        print(os.listdir(os.path.dirname(MERGED_INPUT)))
    else:
        print(f"Directory {os.path.dirname(MERGED_INPUT)} does not exist.")
else:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    for lang in LANGUAGES:
        print(f"\n=== PROCESSING {lang.upper()} ===")
        output_filename = f'bt-{lang}-vanilla.jsonl'
        output_path = os.path.join(OUTPUT_DIR, output_filename)
        
        try:
            # Run with capture_output=True to get stderr in case of failure
            result = subprocess.run([
                sys.executable, '-u', QA_SCRIPT,
                '--mode', 'bt',
                '--lang', lang,
                '--qg_input_path', MERGED_INPUT,
                '--output_path', output_path
            ], capture_output=True, text=True)
            
            if result.returncode != 0:
                print(f"FAILED {lang}. Error Output:")
                print(result.stderr)
                print("Standard Output (last 20 lines):")
                print('\n'.join(result.stdout.splitlines()[-20:]))
                break  # Stop on first failure to avoid cascading errors
            
            print(f"FINISHED {lang}. Saving...")
            if MOUNT_DRIVE and os.path.exists(DRIVE_SAVE_PATH):
                dest_path = os.path.join(DRIVE_SAVE_PATH, output_filename)
                shutil.copy2(output_path, dest_path)
                print(f"Copied to Drive: {dest_path}")
            
            files.download(output_path)
            
        except Exception as e:
            print(f"Unexpected error processing {lang}: {e}")
            import traceback
            traceback.print_exc()